In [2]:
import pandas as pd
import numpy as np

df = pd.read_parquet('cleaned_hospital_data_final.parquet')

In [3]:
sum(df.duplicated())

55

In [4]:
df.isna().sum().sum()

0

In [5]:
for col in df.select_dtypes(include=['object', 'category']).columns:
    blank_mask = df[col].str.strip().eq('').fillna(False)

    if blank_mask.any():
        print(f"Warning: {col} contains {blank_mask.sum()} blank values")

In [6]:
for col in ['Total Charges', 'Total Costs']:
    if df[col].dtype == 'object':
        df[col] = (
            df[col]
            .astype(str)
            .str.replace('$', '', regex=False)
            .str.replace(',', '', regex=False)
            .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Zero or Negative Charges:", (df['Total Charges'] <= 0).sum())
print("Zero or Negative Costs:", (df['Total Costs'] <= 0).sum())

Zero or Negative Charges: 0
Zero or Negative Costs: 5


In [7]:
for col in ['Total Charges', 'Total Costs']:
    print(f"\n{col}")
    print("Zero:", (df[col] == 0).sum())
    print("Negative:", (df[col] < 0).sum())
    print("Missing:", df[col].isna().sum())

problem_rows = df[
    (df['Total Charges'] <= 0) |
    (df['Total Costs'] <= 0) |
    (df['Total Charges'].isna()) |
    (df['Total Costs'].isna())
]

display(problem_rows.head(10))


Total Charges
Zero: 0
Negative: 0
Missing: 0

Total Costs
Zero: 5
Negative: 0
Missing: 0


,Hospital Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code - 3 digits,Gender,Race,Ethnicity,...,Emergency Department Indicator,Total Charges,Total Costs,Is_Neonate,Has_Secondary_Insurance,Cost to Charge Ratio,Cost Charge Difference,Has_Procedure,Procedure Prefix,Procedure Code Grouped
1015527,Finger Lakes,Ontario,3429000,678,F.F. Thompson Hospital,50 to 69,145,M,White,Not Span/Hispanic,...,False,0.01,0.0,0,0,0.0,-0.01,0,No Procedure,No Code
2210496,Hudson Valley,Orange,3523000,699,Garnet Health Medical Center,30 to 49,127,F,White,Not Span/Hispanic,...,False,0.03,0.0,0,0,0.0,-0.03,1,PGN,PGN003
2256374,Hudson Valley,Rockland,4324000,776,Montefiore Nyack,0 to 17,109,F,Other Race,Not Span/Hispanic,...,False,0.01,0.0,1,0,0.0,-0.01,0,No Procedure,No Code
4490671,Hudson Valley,Orange,3523000,699,Garnet Health Medical Center,30 to 49,109,M,White,Not Span/Hispanic,...,False,0.03,0.0,0,0,0.0,-0.03,1,GIS,GIS009
4517986,Hudson Valley,Orange,3523000,699,Garnet Health Medical Center,70 or Older,127,F,White,Not Span/Hispanic,...,False,0.01,0.0,0,1,0.0,-0.01,1,MST,MST006


In [8]:
print("Data type of Length of Stay:", df['Length of Stay'].dtype)
if df['Length of Stay'].dtype == 'object':
    print("Unique non-numeric values:", df[~df['Length of Stay'].astype(str).str.isnumeric()]['Length of Stay'].unique())

Data type of Length of Stay: int32


In [9]:
df['Length of Stay'] = df['Length of Stay'].replace('120 +', '120')

df['Length of Stay'] = pd.to_numeric(df['Length of Stay'], errors='coerce').astype('int32')

print("New Data Type:", df['Length of Stay'].dtype)
print("Max Stay:", df['Length of Stay'].max())
print("Min Stay:", df['Length of Stay'].min())

New Data Type: int32
Max Stay: 120
Min Stay: 1


In [10]:
df['Discharge Year'].value_counts()

Discharge Year
2018    2335280
2019    2321721
2021    2123752
2020    2032994
Name: count, dtype: int64

In [11]:
cols_to_check = ['Gender', 'Race', 'Ethnicity', 'Age Group', 'Type of Admission']
for col in cols_to_check:
    print(f"{col}")
    print(df[col].unique())
    print()

Gender
['F', 'M', 'U']
Categories (3, object): ['F', 'M', 'U']

Race
['White', 'Black/African American', 'Multi-racial', 'Other Race']
Categories (4, object): ['Black/African American', 'Multi-racial', 'Other Race', 'White']

Ethnicity
['Not Span/Hispanic', 'Unknown', 'Spanish/Hispanic', 'Multi-ethnic']
Categories (4, object): ['Multi-ethnic', 'Not Span/Hispanic', 'Spanish/Hispanic', 'Unknown']

Age Group
['50 to 69', '18 to 29', '0 to 17', '30 to 49', '70 or Older']
Categories (5, object): ['0 to 17', '18 to 29', '30 to 49', '50 to 69', '70 or Older']

Type of Admission
['Elective', 'Emergency', 'Newborn', 'Urgent', 'Not Available', 'Trauma']
Categories (6, object): ['Elective', 'Emergency', 'Newborn', 'Not Available', 'Trauma', 'Urgent']



In [12]:
print("Emergency Indicator values:\n", df['Emergency Department Indicator'].value_counts())

Emergency Indicator values:
 Emergency Department Indicator
True     5348736
False    3465011
Name: count, dtype: int64


In [13]:
abnormal_costs = (df['Total Costs'] > df['Total Charges']).sum()
print("Cases where Costs > Charges:", abnormal_costs)

Cases where Costs > Charges: 127140


In [14]:
extreme_cases = df[df['Total Costs'] > (df['Total Charges'] * 5)]
print("Extreme anomalies count:", len(extreme_cases))

Extreme anomalies count: 341


In [15]:
extreme_cases = df[
    df['Total Costs'] > df['Total Charges'] * 5
]

print("Extreme anomalies:", len(extreme_cases))

display(
    extreme_cases[
        ['Total Charges', 'Total Costs', 'Length of Stay',
         'Age Group', 'Type of Admission',
         'Patient Disposition', 'Discharge Year']
    ].sort_values('Total Costs', ascending=False)
)

Extreme anomalies: 341


,Total Charges,Total Costs,Length of Stay,Age Group,Type of Admission,Patient Disposition,Discharge Year
8701132,556847.76,3717778.88,32,50 to 69,Elective,Home or Self Care,2021
8315076,56891.55,3606335.97,2,70 or Older,Emergency,Home w/ Home Health Services,2021
6614224,308147.76,3342323.31,19,50 to 69,Emergency,Home w/ Home Health Services,2020
6664655,113822.26,3292013.12,7,50 to 69,Emergency,Skilled Nursing Home,2020
3584120,477452.56,2395690.38,10,70 or Older,Emergency,Expired,2019
...,...,...,...,...,...,...,...
1725182,4544.00,22967.90,1,0 to 17,Urgent,Home or Self Care,2018
699224,4543.00,22967.79,1,0 to 17,Urgent,Home or Self Care,2018
334884,4320.00,22945.11,1,0 to 17,Urgent,Home or Self Care,2018
4508767,2513.38,16207.68,1,0 to 17,Emergency,Home or Self Care,2019


In [16]:
# How many times Total Costs exceed Total Charges
df['Cost to Charge Ratio'] = (
    df['Total Costs'] / df['Total Charges']
)

# Absolute difference between costs and charges
df['Cost Charge Difference'] = (
    df['Total Costs'] - df['Total Charges']
)

# Show the 10 cases with the highest ratio
display(
    df.nlargest(10, 'Cost to Charge Ratio')[
        [
            'Total Charges',
            'Total Costs',
            'Cost to Charge Ratio',
            'Cost Charge Difference'
        ]
    ]
)

,Total Charges,Total Costs,Cost to Charge Ratio,Cost Charge Difference
8315076,56891.55,3606335.97,63.389659,3549444.42
6664655,113822.26,3292013.12,28.922402,3178190.86
1351190,136934.12,1553062.98,11.341680,1416128.86
2785207,178104.98,1951290.08,10.955842,1773185.10
2047638,179189.19,1951237.60,10.889260,1772048.41
6614224,308147.76,3342323.31,10.846496,3034175.55
2367327,181424.07,1951026.06,10.753954,1769601.99
669156,181686.91,1952086.22,10.744231,1770399.31
1986132,182319.74,1953375.19,10.714008,1771055.45
1142288,182866.97,1952312.82,10.676137,1769445.85


In [17]:
df[[
    'Total Charges',
    'Total Costs',
    'Cost to Charge Ratio',
    'Cost Charge Difference'
        ]].isna().sum().sum()

0

In [18]:
df.isna().sum().sum()

0

In [19]:
df.dropna(inplace=True)

In [20]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8813747 entries, 0 to 8813746
Data columns (total 38 columns):
 #   Column                               Dtype   
---  ------                               -----   
 0   Hospital Service Area                category
 1   Hospital County                      category
 2   Operating Certificate Number         int32   
 3   Permanent Facility Id                int32   
 4   Facility Name                        category
 5   Age Group                            category
 6   Zip Code - 3 digits                  category
 7   Gender                               category
 8   Race                                 category
 9   Ethnicity                            category
 10  Length of Stay                       int32   
 11  Type of Admission                    category
 12  Patient Disposition                  category
 13  Discharge Year                       int16   
 14  CCSR Diagnosis Code                  category
 15  CCSR Diagnosis 

In [21]:
categorical_cols = [
    'Hospital Service Area', 'Hospital County', 'Age Group', 'Zip Code - 3 digits',
    'Gender', 'Race', 'Ethnicity', 'Type of Admission', 'Patient Disposition',
    'APR Severity of Illness Description', 'APR Risk of Mortality',
    'APR Medical Surgical Description', 'Payment Typology 1'
]

for col in categorical_cols:
    df[col] = df[col].astype('category')

df['Discharge Year'] = df['Discharge Year'].astype('int16')
df['APR DRG Code'] = df['APR DRG Code'].astype('int16')
df['APR MDC Code'] = df['APR MDC Code'].astype('int8')
df['APR Severity of Illness Code'] = df['APR Severity of Illness Code'].astype('int8')
df['Is_Neonate'] = df['Is_Neonate'].astype('int8')

df['Operating Certificate Number'] = df['Operating Certificate Number'].astype('int32')
df['Permanent Facility Id'] = df['Permanent Facility Id'].astype('int32')

df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8813747 entries, 0 to 8813746
Data columns (total 38 columns):
 #   Column                               Dtype   
---  ------                               -----   
 0   Hospital Service Area                category
 1   Hospital County                      category
 2   Operating Certificate Number         int32   
 3   Permanent Facility Id                int32   
 4   Facility Name                        category
 5   Age Group                            category
 6   Zip Code - 3 digits                  category
 7   Gender                               category
 8   Race                                 category
 9   Ethnicity                            category
 10  Length of Stay                       int32   
 11  Type of Admission                    category
 12  Patient Disposition                  category
 13  Discharge Year                       int16   
 14  CCSR Diagnosis Code                  category
 15  CCSR Diagnosis 

In [22]:
df.dtypes.value_counts()

int8        5
float64     4
int32       3
category    2
int16       2
category    1
category    1
category    1
bool        1
Float64     1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
Name: count, dtype: int64

In [23]:
object_cols = df.select_dtypes(include=['object']).columns

print("Object columns:", len(object_cols))

for col in object_cols:
    print(f"{col}: {df[col].nunique(dropna=False)} unique values")

Object columns: 0


In [24]:
print(
    "Memory usage (MB):",
    round(df.memory_usage(deep=True).sum() / 1024**2, 2)
)

Memory usage (MB): 765.2


In [25]:
categorical_cols = [
    'Facility Name',
    'CCSR Diagnosis Code',
    'CCSR Diagnosis Description',
    'CCSR Procedure Code',
    'CCSR Procedure Description',
    'APR DRG Description',
    'APR MDC Description',
    'Emergency Department Indicator'
]

for col in categorical_cols:
    df[col] = df[col].astype('category')

print(df[categorical_cols].dtypes)

Facility Name                     category
CCSR Diagnosis Code               category
CCSR Diagnosis Description        category
CCSR Procedure Code               category
CCSR Procedure Description        category
APR DRG Description               category
APR MDC Description               category
Emergency Department Indicator    category
dtype: object


In [26]:
ccsr_map = {
    'BLD': 'Blood Diseases',
    'CIR': 'Circulatory System Diseases',
    'DEN': 'Dental Diseases',
    'DIG': 'Digestive System Diseases',
    'EAR': 'Ear Diseases',
    'END': 'Endocrine, Nutritional, Metabolic Diseases',
    'EXT': 'External Causes of Morbidity',
    'EYE': 'Eye Diseases',
    'FAC': 'Factors Influencing Health Status',
    'GEN': 'Genitourinary System Diseases',
    'INF': 'Infectious and Parasitic Diseases',
    'INJ': 'Injury and Poisoning',
    'MAL': 'Congenital Malformations',
    'MBD': 'Mental, Behavioral, Neurodevelopmental Disorders',
    'MUS': 'Musculoskeletal System Diseases',
    'NEO': 'Neoplasms',
    'NVS': 'Nervous System Diseases',
    'PNL': 'Perinatal Period Conditions',
    'PRG': 'Pregnancy, Childbirth, Puerperium',
    'RSP': 'Respiratory System Diseases',
    'SKN': 'Skin Diseases',
    'SYM': 'Symptoms, Signs, Abnormal Findings',
    'XXX': 'Unacceptable Principal Diagnosis',
}

col = 'CCSR Diagnosis Code'

cats = df[col].cat.categories
prefix = cats.astype(str).str[:3].str.upper()

unmatched = sorted(set(cats[~prefix.isin(ccsr_map.keys())]))
print("Unmatched codes:", unmatched[:20])

new_names = [ccsr_map.get(p, c) for p, c in zip(prefix, cats.astype(str))]

codes = df[col].cat.codes.to_numpy()
new_cats = pd.Index(new_names)
unique_cats = new_cats.unique()
remap = unique_cats.get_indexer(new_cats) 
new_codes = np.where(codes >= 0, remap[codes], -1)

df[col] = pd.Categorical.from_codes(new_codes, categories=unique_cats)

print(df[col].value_counts(dropna=False))

Unmatched codes: ['Blood Diseases', 'Congenital Malformations', 'Mental, Behavioral, Neurodevelopmental Disorders', 'Nervous System Diseases', 'Perinatal Period Conditions', 'Pregnancy, Childbirth, Puerperium', 'Respiratory System Diseases', 'Skin Diseases', 'Unacceptable Principal Diagnosis']
CCSR Diagnosis Code
Circulatory System Diseases                         1188161
Pregnancy, Childbirth, Puerperium                    898437
Perinatal Period Conditions                          831865
Digestive System Diseases                            768496
Infectious and Parasitic Diseases                    739982
Mental, Behavioral, Neurodevelopmental Disorders     705482
Injury and Poisoning                                 635395
Respiratory System Diseases                          559606
Musculoskeletal System Diseases                      470951
Neoplasms                                            394229
Genitourinary System Diseases                        381871
Endocrine, Nutritional, M

In [27]:
df['CCSR Diagnosis Code'].value_counts()

CCSR Diagnosis Code
Circulatory System Diseases                         1188161
Pregnancy, Childbirth, Puerperium                    898437
Perinatal Period Conditions                          831865
Digestive System Diseases                            768496
Infectious and Parasitic Diseases                    739982
Mental, Behavioral, Neurodevelopmental Disorders     705482
Injury and Poisoning                                 635395
Respiratory System Diseases                          559606
Musculoskeletal System Diseases                      470951
Neoplasms                                            394229
Genitourinary System Diseases                        381871
Endocrine, Nutritional, Metabolic Diseases           375154
Nervous System Diseases                              269504
Symptoms, Signs, Abnormal Findings                   203812
Skin Diseases                                        140346
Blood Diseases                                       110404
Factors Influencing 

In [28]:
print(df['Type of Admission'].value_counts(normalize=True))

print(df.groupby('Type of Admission', observed=True)['Length of Stay']
        .agg(['count', 'median', 'mean']))

pd.crosstab(df['Type of Admission'], df['Emergency Department Indicator'],
            normalize='index')

Type of Admission
Emergency        0.651159
Elective         0.177672
Newborn          0.091047
Urgent           0.076273
Trauma           0.003398
Not Available    0.000451
Name: proportion, dtype: float64
                     count  median      mean
Type of Admission                           
Elective           1565956     3.0  4.629481
Emergency          5739155     4.0  5.991698
Newborn             802464     2.0  3.638770
Not Available         3974     3.0  6.256920
Trauma               29951     4.0  6.388134
Urgent              672247     3.0  6.187159


Emergency Department Indicator,False,True
Type of Admission,,
Elective,0.974527,0.025473
Emergency,0.094917,0.905083
Newborn,0.999225,0.000775
Not Available,0.855058,0.144942
Trauma,0.101766,0.898234
Urgent,0.871575,0.128425


In [29]:
import numpy as np
import pandas as pd

col = 'CCSR Procedure Code'
NO_PROC = {'No Code', 'NO CODE', 'No Procedure', ''}
RARE_THRESHOLD = 50

def remap_categories(series, func):
    """بيطبّق func على الـ categories بس (مئات القيم) وبيوزعها على الصفوف."""
    cats = series.cat.categories
    new_vals = pd.Index([func(c) for c in cats])
    uniq = new_vals.unique()
    remap = uniq.get_indexer(new_vals)
    codes = series.cat.codes.to_numpy()
    new_codes = np.where(codes >= 0, remap[codes], -1)
    return pd.Categorical.from_codes(new_codes, categories=uniq)

# 1) Has_Procedure  (NaN أو علامة الغياب = مفيش إجراء)
no_proc_cats = [c for c in df[col].cat.categories if str(c).strip() in NO_PROC]
codes = df[col].cat.codes.to_numpy()
is_missing_code = np.isin(codes, [df[col].cat.categories.get_loc(c) for c in no_proc_cats])
df['Has_Procedure'] = (~((codes < 0) | is_missing_code)).astype('int8')

def to_prefix(c):
    c = str(c).strip()
    return 'No Procedure' if c in NO_PROC else c[:3].upper()

df['Procedure Prefix'] = remap_categories(df[col], to_prefix)

counts = df[col].value_counts()
rare = set(counts[counts < RARE_THRESHOLD].index)

def group_rare(c):
    return 'Other' if c in rare else str(c)

df['Procedure Code Grouped'] = remap_categories(df[col], group_rare)

print(df['Has_Procedure'].value_counts(normalize=True))
print(df['Procedure Prefix'].nunique(), 'prefixes |',
      df['Procedure Code Grouped'].nunique(), 'grouped codes')

print(df.groupby('Procedure Prefix', observed=True)['Length of Stay']
        .agg(['count', 'median', 'mean'])
        .sort_values('median'))

Has_Procedure
1    0.705767
0    0.294233
Name: proportion, dtype: float64
32 prefixes | 315 grouped codes
                    count  median       mean
Procedure Prefix                            
CHP                     2     1.5   1.500000
PNS                 11195     2.0   3.621438
ENP                 12931     2.0   3.832496
ENT                123320     2.0   3.280165
PGN                802659     2.0   2.793437
EYP                  3926     2.0   5.919256
FRS                 67340     2.0   2.969112
MRS                201272     2.0   2.682758
ADM                992640     3.0   5.269102
SKB                 85510     3.0   5.635984
PLC                 11268     3.0   5.098420
No Procedure      2593297     3.0   4.427194
NCM                  2936     3.0   4.232970
MST                722977     3.0   5.226892
MAM                226378     3.0   4.704609
EST                 36820     3.0   5.146795
IMG                240968     3.0   4.945412
OTR                114703     4.0   6.

In [30]:
df

,Hospital Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code - 3 digits,Gender,Race,Ethnicity,...,Emergency Department Indicator,Total Charges,Total Costs,Is_Neonate,Has_Secondary_Insurance,Cost to Charge Ratio,Cost Charge Difference,Has_Procedure,Procedure Prefix,Procedure Code Grouped
0,Capital/Adirond,Columbia,1001000,146,Columbia Memorial Hospital,50 to 69,125,F,White,Not Span/Hispanic,...,True,17591.41,3570.06,0,1,0.202943,-14021.35,0,No Procedure,No Code
1,New York City,Manhattan,7002009,1445,Harlem Hospital Center,50 to 69,100,F,Black/African American,Not Span/Hispanic,...,True,106190.21,78913.24,0,1,0.743131,-27276.97,1,ESA,ESA003
2,New York City,Manhattan,7002009,1445,Harlem Hospital Center,50 to 69,100,F,Black/African American,Not Span/Hispanic,...,False,5600.00,4161.53,0,0,0.743130,-1438.47,1,MHT,MHT001
3,New York City,Kings,7001016,1301,Kings County Hospital Center,18 to 29,112,M,Black/African American,Not Span/Hispanic,...,True,15965.54,11341.30,0,1,0.710361,-4624.24,1,IMG,IMG008
4,New York City,Manhattan,7002054,1464,New York-Presbyterian Hospital - Columbia Pres...,0 to 17,104,M,Multi-racial,Not Span/Hispanic,...,True,36846.83,10240.02,0,1,0.277908,-26606.81,0,No Procedure,No Code
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8813742,Long Island,Suffolk,5153000,913,Huntington Hospital,70 or Older,117,M,White,Not Span/Hispanic,...,True,67029.14,17052.81,0,1,0.254409,-49976.33,1,MST,MST023
8813743,Long Island,Suffolk,5153000,913,Huntington Hospital,70 or Older,117,M,White,Not Span/Hispanic,...,True,34050.80,21813.32,0,1,0.640611,-12237.48,1,ESA,ESA001
8813744,Long Island,Suffolk,5153000,913,Huntington Hospital,70 or Older,117,F,White,Not Span/Hispanic,...,True,32157.37,7719.76,0,1,0.240062,-24437.61,0,No Procedure,No Code
8813745,Long Island,Suffolk,5153000,913,Huntington Hospital,70 or Older,117,F,White,Not Span/Hispanic,...,True,56847.68,15515.27,0,1,0.272927,-41332.41,0,No Procedure,No Code


In [31]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8813747 entries, 0 to 8813746
Data columns (total 38 columns):
 #   Column                               Dtype   
---  ------                               -----   
 0   Hospital Service Area                category
 1   Hospital County                      category
 2   Operating Certificate Number         int32   
 3   Permanent Facility Id                int32   
 4   Facility Name                        category
 5   Age Group                            category
 6   Zip Code - 3 digits                  category
 7   Gender                               category
 8   Race                                 category
 9   Ethnicity                            category
 10  Length of Stay                       int32   
 11  Type of Admission                    category
 12  Patient Disposition                  category
 13  Discharge Year                       int16   
 14  CCSR Diagnosis Code                  category
 15  CCSR Diagnosis 

In [32]:
df.columns

Index(['Hospital Service Area', 'Hospital County',
       'Operating Certificate Number', 'Permanent Facility Id',
       'Facility Name', 'Age Group', 'Zip Code - 3 digits', 'Gender', 'Race',
       'Ethnicity', 'Length of Stay', 'Type of Admission',
       'Patient Disposition', 'Discharge Year', 'CCSR Diagnosis Code',
       'CCSR Diagnosis Description', 'CCSR Procedure Code',
       'CCSR Procedure Description', 'APR DRG Code', 'APR DRG Description',
       'APR MDC Code', 'APR MDC Description', 'APR Severity of Illness Code',
       'APR Severity of Illness Description', 'APR Risk of Mortality',
       'APR Medical Surgical Description', 'Payment Typology 1',
       'Birth Weight', 'Emergency Department Indicator', 'Total Charges',
       'Total Costs', 'Is_Neonate', 'Has_Secondary_Insurance',
       'Cost to Charge Ratio', 'Cost Charge Difference', 'Has_Procedure',
       'Procedure Prefix', 'Procedure Code Grouped'],
      dtype='object')

In [33]:
import pandas as pd

def profile_df(df: pd.DataFrame, table_name: str = "staging_table", sample_n: int = 3):
    n_rows = len(df)
    print(f"Table: {table_name}")
    print(f"Rows: {n_rows:,} | Columns: {df.shape[1]}")
    print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"Duplicate rows: {df.duplicated().sum():,}")
    print("-" * 60)

    rows = []
    for col in df.columns:
        s = df[col]
        non_null = s.dropna()
        nunique = s.nunique(dropna=True)

        info = {
            "column": col,
            "pandas_dtype": str(s.dtype),
            "null_count": int(s.isna().sum()),
            "null_pct": round(s.isna().mean() * 100, 2),
            "distinct_count": nunique,
            "is_unique": nunique == len(non_null) and len(non_null) > 0,
            "sample_values": non_null.unique()[:sample_n].tolist(),
        }

        # numeric metadata
        if pd.api.types.is_numeric_dtype(s):
            info["min"] = non_null.min() if len(non_null) else None
            info["max"] = non_null.max() if len(non_null) else None
            info["is_integer_like"] = bool((non_null % 1 == 0).all()) if len(non_null) else None
            info["max_len"] = None
        # datetime metadata
        elif pd.api.types.is_datetime64_any_dtype(s):
            info["min"] = non_null.min() if len(non_null) else None
            info["max"] = non_null.max() if len(non_null) else None
            info["is_integer_like"] = None
            info["max_len"] = None
        # text / object metadata
        else:
            lens = non_null.astype(str).str.len()
            info["min"] = None
            info["max"] = None
            info["is_integer_like"] = None
            info["max_len"] = int(lens.max()) if len(lens) else None

        info["suggested_sql_type"] = suggest_sql_type(s, info)
        rows.append(info)

    return pd.DataFrame(rows)


def suggest_sql_type(s: pd.Series, info: dict) -> str:
    dt = str(s.dtype)
    if pd.api.types.is_bool_dtype(s):
        return "BIT"
    if pd.api.types.is_integer_dtype(s):
        mx, mn = info["max"], info["min"]
        if mn is not None and mn >= -32768 and mx <= 32767:
            return "SMALLINT"
        if mn is not None and mn >= -2**31 and mx <= 2**31 - 1:
            return "INT"
        return "BIGINT"
    if pd.api.types.is_float_dtype(s):
        return "INT" if info.get("is_integer_like") else "DECIMAL(18,4)"
    if pd.api.types.is_datetime64_any_dtype(s):
        return "DATETIME"
    # object / string
    ml = info.get("max_len") or 1
    size = 50 if ml <= 50 else 100 if ml <= 100 else 255 if ml <= 255 else None
    return f"VARCHAR({size})" if size else "VARCHAR(MAX)"


# ---------- الاستخدام ----------
meta = profile_df(df, table_name="stg_my_table")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)
print(meta.to_string(index=False))

# لو عاوز تحفظ الـ metadata
# meta.to_csv("stg_metadata.csv", index=False)

Table: stg_my_table
Rows: 8,813,747 | Columns: 38
Memory: 765.20 MB
Duplicate rows: 55
------------------------------------------------------------
                             column pandas_dtype  null_count  null_pct  distinct_count  is_unique                                                                                                                   sample_values          min          max is_integer_like  max_len suggested_sql_type
              Hospital Service Area     category           0       0.0               8      False                                                                                 [Capital/Adirond, New York City, Hudson Valley]          NaN          NaN            None     15.0        VARCHAR(50)
                    Hospital County     category           0       0.0              57      False                                                                                                    [Columbia, Manhattan, Kings]          NaN          NaN         

In [36]:
import pandas as pd

src = r"E:\DEPI\Tech\Final Project\cleaned_hospital_data.csv"
dst = r"E:\DEPI\Tech\Final Project\hospital_pipe.csv"

df = pd.read_csv(src)
print(df.shape)   # لازم يطلع (8813747, 38)

df.to_csv(dst, index=False, sep="|", encoding="utf-8", lineterminator="\n")

(8813747, 38)


In [37]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8813747 entries, 0 to 8813746
Data columns (total 38 columns):
 #   Column                               Dtype  
---  ------                               -----  
 0   Hospital Service Area                object 
 1   Hospital County                      object 
 2   Operating Certificate Number         int64  
 3   Permanent Facility Id                int64  
 4   Facility Name                        object 
 5   Age Group                            object 
 6   Zip Code - 3 digits                  object 
 7   Gender                               object 
 8   Race                                 object 
 9   Ethnicity                            object 
 10  Length of Stay                       int64  
 11  Type of Admission                    object 
 12  Patient Disposition                  object 
 13  Discharge Year                       int64  
 14  CCSR Diagnosis Code                  object 
 15  CCSR Diagnosis Description      

In [34]:
df.to_parquet('cleaned_hospital_data_final.parquet', index=False)

In [35]:
df.to_csv('cleaned_hospital_data.csv', index=False)
print("CSV file created successfully!")

CSV file created successfully!
